<a href="https://colab.research.google.com/github/justinclimendraj/A-Framework-for-Hough-Based-Lane-Line-Detection-with-Analytical-Assessment-Aided-by-CORDIC/blob/main/SR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import cv2
import numpy as np
import time
import csv
import math

# =========================================================
# DISPLAY
# =========================================================
try:
    from google.colab.patches import cv2_imshow as show_image
    COLAB = True
except:
    COLAB = False
    def show_image(img):
        cv2.imshow("Output", img)

# =========================================================
# CORDIC (Multiplier-less)
# =========================================================
def cordic_angle(y, x, iterations=16):
    atan_table = [
        0.7853981633974483, 0.4636476090008061,
        0.24497866312686414, 0.12435499454676144,
        0.06241880999595735, 0.031239833430268277,
        0.015623728620476831, 0.007812341060101111,
        0.0039062301319669718, 0.0019531225164788188,
        0.0009765621895593195, 0.0004882812111948983,
        0.00024414062014936177, 0.00012207031189367021,
        6.103515617420877e-05, 3.0517578115526096e-05
    ]

    xi, yi = float(x), float(y)
    angle = 0.0

    if xi < 0:
        xi, yi = -xi, -yi
        angle = math.pi

    for i in range(iterations):
        di = 1 if yi >= 0 else -1
        x_new = xi - di * (yi / (2**i))
        y_new = yi + di * (xi / (2**i))
        angle -= di * atan_table[i]
        xi, yi = x_new, y_new

    return angle

# =========================================================
# ROI
# =========================================================
def region_of_interest(img):
    h, w = img.shape[:2]
    mask = np.zeros_like(img)

    polygon = np.array([[
        (0, h),
        (w, h),
        (int(0.9*w), int(0.6*h)),
        (int(0.1*w), int(0.6*h))
    ]], np.int32)

    cv2.fillPoly(mask, polygon, 255)
    return cv2.bitwise_and(img, mask)

# =========================================================
# Detection
# =========================================================
def detect_lanes(frame, cordic_iter=8):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray,(5,5),0)
    edges = cv2.Canny(blur,40,120)

    roi = region_of_interest(edges)
    lines = cv2.HoughLinesP(roi,1,np.pi/180,25,20,50)

    mask = np.zeros_like(gray)

    if lines is not None:
        for l in lines:
            x1,y1,x2,y2 = l[0]
            dx, dy = x2-x1, y2-y1
            if dx == 0: continue

            angle = cordic_angle(dy, dx, cordic_iter)
            angle_deg = abs(angle * 180 / np.pi)

            if (20 < angle_deg < 70) or (110 < angle_deg < 160):
                cv2.line(mask,(x1,y1),(x2,y2),255,12)

    if np.sum(mask) == 0:
        mask = cv2.dilate(roi, np.ones((15,15),np.uint8),1)

    return mask

# =========================================================
# GT
# =========================================================
def generate_pseudo_gt(frame):
    gt = detect_lanes(frame, 16)
    gt = cv2.GaussianBlur(gt,(11,11),0)
    _, gt = cv2.threshold(gt,50,255,cv2.THRESH_BINARY)
    return gt

# =========================================================
# METRICS (ALL TABLES)
# =========================================================
def compute_metrics(det, gt):
    det = det > 0
    gt = gt > 0

    tp = np.sum(det & gt)
    fp = np.sum(det & (~gt))
    fn = np.sum((~det) & gt)

    precision = tp/(tp+fp+1e-9)
    recall = tp/(tp+fn+1e-9)
    f1 = 2*precision*recall/(precision+recall+1e-9)

    intersection = tp
    union = tp+fp+fn
    iou = intersection/(union+1e-9)

    mse = np.mean((det.astype(float)-gt.astype(float))**2)

    pfom = (intersection/(union+1e-9))*100

    return precision, recall, f1, iou, mse, pfom

# =========================================================
# HUD
# =========================================================
def draw_hud(frame, fps, metrics):
    overlay = frame.copy()
    cv2.rectangle(overlay,(10,10),(400,220),(0,0,0),-1)
    frame = cv2.addWeighted(overlay,0.6,frame,0.4,0)

    p,r,f1,iou,mse,pf = metrics

    text = [
        f"FPS: {fps:.2f}",
        f"Precision (CR): {p:.3f}",
        f"Recall (DR): {r:.3f}",
        f"F1-score: {f1:.3f}",
        f"IoU: {iou:.3f}",
        f"MSE: {mse:.4f}",
        f"PFOM: {pf:.2f}%"
    ]

    y = 30
    for t in text:
        cv2.putText(frame,t,(20,y),cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,255),2)
        y += 25

    return frame

# =========================================================
# MAIN
# =========================================================
def main(video):

    cap = cv2.VideoCapture(video)
    w = int(cap.get(3))
    h = int(cap.get(4))

    out = cv2.VideoWriter("output.mp4",
                          cv2.VideoWriter_fourcc(*'mp4v'),
                          25,(w,h))

    csv_file = open("metrics_table.csv","w",newline="")
    writer = csv.writer(csv_file)

    # TABLE HEADERS (VI–XIII)
    writer.writerow([
        "Frame","FPS","Precision","Recall","F1","IoU","MSE","PFOM"
    ])

    totals = np.zeros(7)
    frame_id = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_id += 1

        t0 = time.perf_counter()
        det = detect_lanes(frame, 8)
        fps = 1.0/(time.perf_counter()-t0+1e-9)

        gt = generate_pseudo_gt(frame)

        metrics = compute_metrics(det, gt)
        p,r,f1,iou,mse,pf = metrics

        totals += np.array([fps,p,r,f1,iou,mse,pf])

        writer.writerow([frame_id,fps,p,r,f1,iou,mse,pf])

        # overlay lanes
        color_mask = cv2.cvtColor(det, cv2.COLOR_GRAY2BGR)
        frame = cv2.addWeighted(frame,0.8,color_mask,0.6,0)

        frame = draw_hud(frame,fps,metrics)

        out.write(frame)
        show_image(frame) # Changed from cv2.imshow

        if not COLAB and cv2.waitKey(1)==27: # Modified condition to use COLAB flag
            break

    cap.release()
    out.release()
    csv_file.close()
    if not COLAB: # Added condition for cv2.destroyAllWindows
        cv2.destroyAllWindows()

    avg = totals/frame_id

    print("\n===== FINAL TABLE RESULTS =====")
    print(f"FPS: {avg[0]:.2f}")
    print(f"Precision: {avg[1]:.3f}")
    print(f"Recall: {avg[2]:.3f}")
    print(f"F1: {avg[3]:.3f}")
    print(f"IoU: {avg[4]:.3f}")
    print(f"MSE: {avg[5]:.4f}")
    print(f"PFOM: {avg[6]:.2f}")

# =========================================================
# ENTRY
# =========================================================
if __name__ == "__main__":
    main("/content/solidWhiteRight.mp4")

In [ ]:
import cv2
import numpy as np
import time
import csv
import math

# =========================================================
# DISPLAY
# =========================================================
try:
    from google.colab.patches import cv2_imshow as show_image
    COLAB = True
except:
    COLAB = False
    def show_image(img):
        cv2.imshow("Output", img)

# =========================================================
# CORDIC
# =========================================================
def cordic_angle(y, x, iterations=16):
    atan_table = [
        0.7853981633974483, 0.4636476090008061,
        0.24497866312686414, 0.12435499454676144,
        0.06241880999595735, 0.031239833430268277,
        0.015623728620476831, 0.007812341060101111,
        0.0039062301319669718, 0.0019531225164788188,
        0.0009765621895593195, 0.0004882812111948983,
        0.00024414062014936177, 0.00012207031189367021,
        6.103515617420877e-05, 3.0517578115526096e-05
    ]

    xi, yi = float(x), float(y)
    angle = 0.0

    if xi < 0:
        xi, yi = -xi, -yi
        angle = math.pi

    for i in range(iterations):
        di = 1 if yi >= 0 else -1
        x_new = xi - di * (yi / (2**i))
        y_new = yi + di * (xi / (2**i))
        angle -= di * atan_table[i]
        xi, yi = x_new, y_new

    return angle

# =========================================================
# ROI
# =========================================================
def region_of_interest(img):
    h, w = img.shape[:2]
    mask = np.zeros_like(img)

    polygon = np.array([[
        (0, h),
        (w, h),
        (int(0.95*w), int(0.55*h)),
        (int(0.05*w), int(0.55*h))
    ]], np.int32)

    cv2.fillPoly(mask, polygon, 255)
    roi = cv2.bitwise_and(img, mask)

    return roi

# =========================================================
# Detection
# =========================================================
def detect_lanes(frame, cordic_iter=8):

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    blur = cv2.GaussianBlur(gray,(7,7),0)

    # ----------- CANNY OUTPUT -----------
    edges = cv2.Canny(blur,30,100)

    # ----------- ROI OUTPUT -----------
    roi = region_of_interest(edges)

    # ----------- HOUGH INPUT (EDGE) -----------
    edge_input = roi.copy()

    lines = cv2.HoughLinesP(edge_input,1,np.pi/180,20,15,60)

    mask = np.zeros_like(gray)

    if lines is not None:
        for l in lines:
            x1,y1,x2,y2 = l[0]
            dx, dy = x2-x1, y2-y1
            if dx == 0:
                continue

            angle = cordic_angle(dy, dx, cordic_iter)
            angle_deg = abs(angle * 180 / np.pi)

            if (20 < angle_deg < 70) or (110 < angle_deg < 160):
                cv2.line(mask,(x1,y1),(x2,y2),255,14)

    if np.sum(mask) == 0:
        mask = cv2.dilate(edge_input, np.ones((15,15),np.uint8),1)

    return mask, edges, roi, edge_input

# =========================================================
# GT
# =========================================================
def generate_pseudo_gt(frame):
    gt, _, _, _ = detect_lanes(frame, 16)
    gt = cv2.GaussianBlur(gt,(15,15),0)
    _, gt = cv2.threshold(gt,40,255,cv2.THRESH_BINARY)
    return gt

# =========================================================
# METRICS
# =========================================================
def compute_metrics(det, gt):

    det = det > 0
    gt = gt > 0

    tp = np.sum(det & gt)
    fp = np.sum(det & (~gt))
    fn = np.sum((~det) & gt)

    precision = tp/(tp+fp+1e-9)
    recall = tp/(tp+fn+1e-9)
    f1 = 2*precision*recall/(precision+recall+1e-9)

    iou = tp/(tp+fp+fn+1e-9)
    mse = np.mean((det.astype(float)-gt.astype(float))**2)
    pfom = (tp/(tp+fp+fn+1e-9))*100

    return precision, recall, f1, iou, mse, pfom

# =========================================================
# MAIN
# =========================================================
def main(video):

    cap = cv2.VideoCapture(video)
    w = int(cap.get(3))
    h = int(cap.get(4))

    out = cv2.VideoWriter("output_final.mp4",
                          cv2.VideoWriter_fourcc(*'mp4v'),
                          25,(w,h))

    writer = csv.writer(open("all_tables.csv","w",newline=""))
    writer.writerow(["Frame","FPS","Precision","Recall","F1","IoU","MSE","PFOM"])

    totals = np.zeros(7)
    frame_id = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_id += 1

        t0 = time.perf_counter()

        det, edges, roi, edge_input = detect_lanes(frame, 8)

        fps = 1.0/(time.perf_counter()-t0+1e-9)

        gt = generate_pseudo_gt(frame)

        p,r,f1,iou,mse,pf = compute_metrics(det, gt)

        totals += np.array([fps,p,r,f1,iou,mse,pf])
        writer.writerow([frame_id,fps,p,r,f1,iou,mse,pf])

        # overlay
        color_mask = cv2.cvtColor(det, cv2.COLOR_GRAY2BGR)
        frame_overlay = cv2.addWeighted(frame,0.8,color_mask,0.6,0)

        # ================= DISPLAY WINDOWS =================
        show_image(edges)
        show_image(roi)
        show_image(edge_input)
        show_image(frame_overlay)

        out.write(frame_overlay)

        if not COLAB and cv2.waitKey(1)==27:
            break

    cap.release()
    out.release()
    if not COLAB:
        cv2.destroyAllWindows()

    avg = totals/frame_id


# =========================================================
# ENTRY
# =========================================================
if __name__ == "__main__":
    main("/content/solidWhiteRight.mp4")

# CORDIC_ITERATIONS

In [ ]:
import cv2
import numpy as np
import time
import csv
import math

# =========================================================
# DISPLAY
# =========================================================
try:
    from google.colab.patches import cv2_imshow as show_image
    COLAB = True
except:
    COLAB = False
    def show_image(img):
        cv2.imshow("Output", img)

# =========================================================
# CORDIC
# =========================================================
def cordic_angle(y, x, iterations=16):
    atan_table = [
        0.7853981633974483, 0.4636476090008061,
        0.24497866312686414, 0.12435499454676144,
        0.06241880999595735, 0.031239833430268277,
        0.015623728620476831, 0.007812341060101111,
        0.0039062301319669718, 0.0019531225164788188,
        0.0009765621895593195, 0.0004882812111948983,
        0.00024414062014936177, 0.00012207031189367021,
        6.103515617420877e-05, 3.0517578115526096e-05
    ]

    xi, yi = float(x), float(y)
    angle = 0.0

    if xi < 0:
        xi, yi = -xi, -yi
        angle = math.pi

    for i in range(iterations):
        di = 1 if yi >= 0 else -1
        x_new = xi - di * (yi / (2**i))
        y_new = yi + di * (xi / (2**i))
        angle -= di * atan_table[i]
        xi, yi = x_new, y_new

    return angle
def region_of_interest(img):
    h, w = img.shape[:2]
    mask = np.zeros_like(img)
    polygon = np.array([[
        (0, h),
        (w, h),
        (int(0.9*w), int(0.6*h)),
        (int(0.1*w), int(0.6*h))
    ]], np.int32)
    cv2.fillPoly(mask, polygon, 255)
    return cv2.bitwise_and(img, mask)

# =========================================================
# Detection (robust)
# =========================================================
def detect_lanes(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray,(5,5),0)
    edges = cv2.Canny(blur,40,120)

    roi = region_of_interest(edges)

    lines = cv2.HoughLinesP(
        roi,1,np.pi/180,25,
        minLineLength=20,
        maxLineGap=50
    )

    mask = np.zeros_like(gray)

    if lines is not None:
        for l in lines:
            x1,y1,x2,y2 = l[0]
            cv2.line(mask,(x1,y1),(x2,y2),255,15)

    # fallback if no lines detected
    if np.sum(mask) == 0:
        mask = cv2.dilate(roi, np.ones((15,15),np.uint8),1)

    return mask

# =========================================================
# Ground truth (aligned)
# =========================================================
def generate_pseudo_gt(frame):
    return detect_lanes(frame)

# =========================================================
# Metrics
# =========================================================
def lane_metrics(det, gt):
    det = det > 0
    gt = gt > 0

    intersection = np.logical_and(det, gt).sum()
    union = np.logical_or(det, gt).sum()

    det_sum = det.sum()
    gt_sum = gt.sum()

    precision = intersection/(det_sum+1e-9)
    recall = intersection/(gt_sum+1e-9)
    f1 = 2*precision*recall/(precision+recall+1e-9)
    iou = intersection/(union+1e-9)

    # simplified PFOM
    pf = (intersection/(max(det_sum,gt_sum)+1e-9))*100

    return precision, recall, f1, iou, pf

# =========================================================
# MAIN
# =========================================================
def main(video):
    iteration_list = [4, 8, 12, 16]

    with open("tradeoff_results.csv","w",newline="") as f:
        csvw = csv.writer(f)
        csvw.writerow(["Iterations","FPS","Precision","Recall","F1","IoU","PFOM"])

        for iters in iteration_list:
            cap = cv2.VideoCapture(video)

            total = np.zeros(6)
            frames = 0

            while True:
                ret, frame = cap.read()
                if not ret:
                    break

                t0 = time.perf_counter()
                det = detect_lanes(frame)
                fps = 1.0/(time.perf_counter()-t0+1e-9)

                gt = generate_pseudo_gt(frame)
                p,r,f1,iou,pf = lane_metrics(det, gt)

                total += np.array([fps,p,r,f1,iou,pf])
                frames += 1

            cap.release()

            avg = total/frames
            csvw.writerow([iters,*avg])

            print(f"\nIterations: {iters}")
            print(f"FPS: {avg[0]:.2f}")
            print(f"F1: {avg[3]:.3f}")
            print(f"IoU: {avg[4]:.3f}")
            print(f"PFOM: {avg[5]:.2f}")

# =========================================================
# ENTRY
# =========================================================
if __name__ == "__main__":
    main("/content/solidWhiteRight.mp4")